In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib
import shap
from math import sqrt
from keras.callbacks import ModelCheckpoint
from keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from imblearn.over_sampling import SMOTE
from sklearn import svm
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from matplotlib import font_manager, rc
path = "c:/Windows/Fonts/malgun.ttf"
if platform.system() == 'Darwin':
    rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    font_name = font_manager.FontProperties(fname=path).get_name()
    rc('font', family=font_name)
else:
    print('Unknown system... sorry~~~~')

C:\Anaconda3\lib\site-packages\statsmodels\tools\_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm
Using TensorFlow backend.


In [79]:
df = pd.read_csv("C:/Users/PC/Desktop/찐_재해데이터_전국팩토리온기업수_결합데이터.csv", encoding = "cp949")
df.shape

Columns (66) have mixed types.Specify dtype option on import or set low_memory=False.


(13707, 189)

In [80]:
df['기업규모'] = 0
df.loc[data['근로자수'] >= 5, '기업규모'] = 1
df.loc[data['근로자수'] >= 50, '기업규모'] = 2
df.loc[data['근로자수'] >= 100, '기업규모'] = 3
df.loc[data['근로자수'] >= 300, '기업규모'] = 4
df.loc[data['근로자수'] >= 1000, '기업규모'] = 5
#data.drop(columns = ["근로자수"], inplace = True)
df.head()

,id,left,top,right,bottom,그리드 내 총 기업 수,연번,시도명,시군구명,관리기관,...,평균급여액,평균급여_1,평균급여_2,평균급여_3,평균급여_4,평균급여_5,평균급여_6,위도,경도,재해발생년도
0,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4251435.0,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,1998
1,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4251435.0,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,2018
2,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4251435.0,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,2021
3,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4251435.0,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,2019
4,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4251435.0,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,1991


In [81]:
df.매출액 = 0
for i, j in enumerate(df.재해발생년도.values, 0):
    if j == 2016:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2016"]
    elif j == 2017:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2017"]
    elif j == 2018:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2018"]
    elif j == 2019:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2019"]
    elif j == 2020:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2020"]
    elif j == 2021:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2021"]
    else:
        df.loc[i, "매출액"] = 0

In [82]:
df[df["매출액"] == 0]

,id,left,top,right,bottom,그리드 내 총 기업 수,연번,시도명,시군구명,관리기관,...,평균급여_1,평균급여_2,평균급여_3,평균급여_4,평균급여_5,평균급여_6,위도,경도,재해발생년도,매출액
0,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,1998,0.0
4,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4398875.0,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,1991,0.0
5,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30841,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4451744.0,4620961.0,4745184.0,4920405.0,5146259.0,5123706.0,35.527824,129.394562,2002,0.0
9,3246187,178508.7749,424587.6958,178608.7749,424487.6958,14,49558,경기도,NaN,한국산업단지공단 시화지사 시화MTV사무소,...,NaN,NaN,NaN,NaN,NaN,NaN,37.319634,126.758054,2019,0.0
19,3246195,178508.7749,423787.6958,178608.7749,423687.6958,5,44962,경기도,NaN,한국산업단지공단 경기지역본부,...,3575322.0,3487896.0,3624239.0,3849586.0,NaN,NaN,37.312804,126.758628,2020,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13662,2983313,173908.7749,432187.6958,174008.7749,432087.6958,3,20658,인천광역시,NaN,한국산업단지공단 인천지역본부,...,NaN,NaN,NaN,NaN,NaN,NaN,37.388493,126.705468,2020,0.0
13663,2983313,173908.7749,432187.6958,174008.7749,432087.6958,3,20658,인천광역시,NaN,한국산업단지공단 인천지역본부,...,NaN,NaN,NaN,NaN,NaN,NaN,37.388493,126.705468,2018,0.0
13704,16616605,412508.7749,224787.6958,412608.7749,224687.6958,4,30063,울산광역시,NaN,한국산업단지공단 울산지역본부,...,3035594.0,3086581.0,3139357.0,3342007.0,3317378.0,3342667.0,35.497014,129.342726,2019,0.0
13705,16616605,412508.7749,224787.6958,412608.7749,224687.6958,4,30063,울산광역시,NaN,한국산업단지공단 울산지역본부,...,3035594.0,3086581.0,3139357.0,3342007.0,3317378.0,3342667.0,35.497014,129.342726,2021,0.0


In [83]:
grouped = df.groupby(by=['대업종', '기업규모'])
df_grouped = grouped.매출액.median().unstack()
df_grouped

기업규모,0,1,2,3,4,5
대업종,,,,,,
건설업,3423453.5,4861100.0,1.460808e+08,3.604248e+07,8.890992e+09,1.014844e+10
기타의사업,1736374.5,5760494.5,2.155992e+09,2.369243e+09,2.142282e+08,1.756484e+09
운수·창고·통신업,0.0,19939392.0,3.563246e+07,5.088755e+08,4.139059e+08,NaN
전기·가스·증기및수도사업,NaN,46946330.0,3.696689e+08,6.590876e+08,NaN,NaN
제조업,557908.5,3730933.0,2.236227e+07,7.103939e+07,4.213202e+08,2.999488e+09


In [84]:
df_0_제조업 = df_grouped.iloc[4,0]
df_1_제조업 = df_grouped.iloc[4,1]
df_2_제조업 = df_grouped.iloc[4,2]
df_3_제조업 = df_grouped.iloc[4,3]
df_4_제조업 = df_grouped.iloc[4,4]
df_5_제조업 = df_grouped.iloc[4,5]

df_0_건설업 = df_grouped.iloc[0,0]
df_1_건설업 = df_grouped.iloc[0,1]
df_2_건설업 = df_grouped.iloc[0,2]
df_3_건설업 = df_grouped.iloc[0,3]
df_4_건설업 = df_grouped.iloc[0,4]
df_5_건설업 = df_grouped.iloc[0,5]

df_0_기타의사업 = df_grouped.iloc[1,0]
df_1_기타의사업 = df_grouped.iloc[1,1]
df_2_기타의사업 = df_grouped.iloc[1,2]
df_3_기타의사업 = df_grouped.iloc[1,3]
df_4_기타의사업 = df_grouped.iloc[1,4]
df_5_기타의사업 = df_grouped.iloc[1,5]

df_0_운수·창고·통신업 = 19939392.0	
df_1_운수·창고·통신업 = df_grouped.iloc[2,1]
df_2_운수·창고·통신업 = df_grouped.iloc[2,2]
df_3_운수·창고·통신업 = df_grouped.iloc[2,3]
df_4_운수·창고·통신업 = df_grouped.iloc[2,4]
df_5_운수·창고·통신업 = df_grouped.iloc[2,5]

In [85]:
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_제조업

df.loc[(df.대업종 == '건설업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_건설업

df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_기타의사업

df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 0)&(df["매출액"] == 0), '매출액'] = df_0_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 1)&(df["매출액"] == 0), '매출액'] = df_1_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 2)&(df["매출액"] == 0), '매출액'] = df_2_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 3)&(df["매출액"] == 0), '매출액'] = df_3_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 4)&(df["매출액"] == 0), '매출액'] = df_4_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 5)&(df["매출액"] == 0), '매출액'] = df_5_운수·창고·통신업

In [86]:
df[df["매출액"] == 0]

,id,left,top,right,bottom,그리드 내 총 기업 수,연번,시도명,시군구명,관리기관,...,평균급여_1,평균급여_2,평균급여_3,평균급여_4,평균급여_5,평균급여_6,위도,경도,재해발생년도,매출액


In [87]:
df.부채비율 = 0
for i, j in enumerate(df.재해발생년도.values, 0):
    if j == 2016:
        df.loc[i, "부채비율"] = df.loc[i, "부채비율_2016"]
    elif j == 2017:
        df.loc[i, "부채비율"] = df.loc[i, "부채비율_2017"]
    elif j == 2018:
        df.loc[i, "부채비율"] = df.loc[i, "부채비율_2018"]
    elif j == 2019:
        df.loc[i, "부채비율"] = df.loc[i, "부채비율_2019"]
    elif j == 2020:
        df.loc[i, "부채비율"] = df.loc[i, "부채비율_2020"]
    elif j == 2021:
        df.loc[i, "부채비율"] = df.loc[i, "부채비율_2021"]
    else:
        df.loc[i, "부채비율"] = 0

In [88]:
df[df["부채비율"] == 0]

,id,left,top,right,bottom,그리드 내 총 기업 수,연번,시도명,시군구명,관리기관,...,평균급여_2,평균급여_3,평균급여_4,평균급여_5,평균급여_6,위도,경도,재해발생년도,매출액,부채비율
0,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,1998,7.103939e+07,0.0
4,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30324,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4566693.0,4720287.0,4879782.0,4972776.0,4970447.0,35.527824,129.394562,1991,5.579085e+05,0.0
5,16879367,417108.7749,228387.6958,417208.7749,228287.6958,2,30841,울산광역시,NaN,한국산업단지공단 울산지역본부,...,4620961.0,4745184.0,4920405.0,5146259.0,5123706.0,35.527824,129.394562,2002,2.155992e+09,0.0
86,12949187,348308.7749,191987.6958,348408.7749,191887.6958,3,59048,경상남도,NaN,한국산업단지공단 경남지역본부,...,4591806.0,4685510.0,4735716.0,4982604.0,4948217.0,35.212871,128.629280,2012,4.213202e+08,0.0
344,13217703,353008.7749,191487.6958,353108.7749,191387.6958,2,59044,경상남도,NaN,한국산업단지공단 경남지역본부,...,3386731.0,3694691.0,3997175.0,3973226.0,4033771.0,35.207789,128.681092,2015,7.103939e+07,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13287,14017650,367008.7749,178787.6958,367108.7749,178687.6958,3,16075,부산광역시,NaN,한국산업단지공단 부산지역본부,...,3259367.0,3450732.0,3378667.0,3608424.0,3599389.0,35.091342,128.831483,1995,3.730933e+06,0.0
13334,3269034,178908.7749,425087.6958,179008.7749,424987.6958,4,46254,경기도,NaN,한국산업단지공단 경기지역본부,...,2071292.0,2080544.0,2188405.0,2164167.0,2046993.0,37.324067,126.762687,2007,2.236227e+07,0.0
13349,3269043,178908.7749,424187.6958,179008.7749,424087.6958,1,46634,경기도,NaN,한국산업단지공단 경기지역본부,...,3905119.0,4120635.0,4242977.0,4352890.0,4352913.0,37.316543,126.763032,2015,7.103939e+07,0.0
13350,3269043,178908.7749,424187.6958,179008.7749,424087.6958,1,46634,경기도,NaN,한국산업단지공단 경기지역본부,...,3905119.0,4120635.0,4242977.0,4352890.0,4352913.0,37.316543,126.763032,2015,7.103939e+07,0.0


In [89]:
df.기업규모

0        3
1        3
2        3
3        2
4        0
        ..
13702    3
13703    3
13704    1
13705    1
13706    1
Name: 기업규모, Length: 13707, dtype: int64

In [90]:
grouped2 = df.groupby(by=['대업종', '기업규모'])
df_grouped2 = grouped.부채비율.median().unstack()
df_grouped2

기업규모,0,1,2,3,4,5
대업종,,,,,,
건설업,92.0034,65.88855,146.10480,187.53640,168.57060,193.5422
기타의사업,135.3484,115.92795,-832.28380,22.96395,123.26190,146.3995
운수·창고·통신업,0.0000,100.85675,109.13885,83.92260,189.10485,NaN
전기·가스·증기및수도사업,NaN,636.08430,370.48090,177.91310,NaN,NaN
제조업,196.6834,164.36680,113.62550,103.11580,128.82840,157.4180


In [91]:
df_0_제조업 = df_grouped2.iloc[4,0]
df_1_제조업 = df_grouped2.iloc[4,1]
df_2_제조업 = df_grouped2.iloc[4,2]
df_3_제조업 = df_grouped2.iloc[4,3]
df_4_제조업 = df_grouped2.iloc[4,4]
df_5_제조업 = df_grouped2.iloc[4,5]

df_0_건설업 = df_grouped2.iloc[0,0]
df_1_건설업 = df_grouped2.iloc[0,1]
df_2_건설업 = df_grouped2.iloc[0,2]
df_3_건설업 = df_grouped2.iloc[0,3]
df_4_건설업 = df_grouped2.iloc[0,4]
df_5_건설업 = df_grouped2.iloc[0,5]

df_0_기타의사업 = df_grouped2.iloc[1,0]
df_1_기타의사업 = df_grouped2.iloc[1,1]
df_2_기타의사업 = df_grouped2.iloc[1,2]
df_3_기타의사업 = df_grouped2.iloc[1,3]
df_4_기타의사업 = df_grouped2.iloc[1,4]
df_5_기타의사업 = df_grouped2.iloc[1,5]

df_0_운수·창고·통신업 = 100.85675
df_1_운수·창고·통신업 = df_grouped2.iloc[2,1]
df_2_운수·창고·통신업 = df_grouped2.iloc[2,2]
df_3_운수·창고·통신업 = df_grouped2.iloc[2,3]
df_4_운수·창고·통신업 = df_grouped2.iloc[2,4]
df_5_운수·창고·통신업 = df_grouped2.iloc[2,5]

In [92]:
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 0)&(df["부채비율"].isnull()), '부채비율'] = df_0_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 1)&(df["부채비율"].isnull()), '부채비율'] = df_1_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 2)&(df["부채비율"].isnull()), '부채비율'] = df_2_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 3)&(df["부채비율"].isnull()), '부채비율'] = df_3_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 4)&(df["부채비율"].isnull()), '부채비율'] = df_4_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 5)&(df["부채비율"].isnull()), '부채비율'] = df_5_제조업

df.loc[(df.대업종 == '건설업')&(df.기업규모 == 0)&(df["부채비율"].isnull()), '부채비율'] = df_0_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 1)&(df["부채비율"].isnull()), '부채비율'] = df_1_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 2)&(df["부채비율"].isnull()), '부채비율'] = df_2_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 3)&(df["부채비율"].isnull()), '부채비율'] = df_3_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 4)&(df["부채비율"].isnull()), '부채비율'] = df_4_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 5)&(df["부채비율"].isnull()), '부채비율'] = df_5_건설업

df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 0)&(df["부채비율"].isnull()), '부채비율'] = df_0_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 1)&(df["부채비율"].isnull()), '부채비율'] = df_1_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 2)&(df["부채비율"].isnull()), '부채비율'] = df_2_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 3)&(df["부채비율"].isnull()), '부채비율'] = df_3_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 4)&(df["부채비율"].isnull()), '부채비율'] = df_4_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 5)&(df["부채비율"].isnull()), '부채비율'] = df_5_기타의사업

df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 0)&(df["부채비율"].isnull()), '부채비율'] = df_0_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 1)&(df["부채비율"].isnull()), '부채비율'] = df_1_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 2)&(df["부채비율"].isnull()), '부채비율'] = df_2_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 3)&(df["부채비율"].isnull()), '부채비율'] = df_3_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 4)&(df["부채비율"].isnull()), '부채비율'] = df_4_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 5)&(df["부채비율"].isnull()), '부채비율'] = df_5_운수·창고·통신업

In [93]:
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 0)&(df["부채비율"] == 0), '부채비율'] = df_0_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 1)&(df["부채비율"] == 0), '부채비율'] = df_1_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 2)&(df["부채비율"] == 0), '부채비율'] = df_2_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 3)&(df["부채비율"] == 0), '부채비율'] = df_3_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 4)&(df["부채비율"] == 0), '부채비율'] = df_4_제조업
df.loc[(df.대업종 == '제조업')&(df.기업규모 == 5)&(df["부채비율"] == 0), '부채비율'] = df_5_제조업

df.loc[(df.대업종 == '건설업')&(df.기업규모 == 0)&(df["부채비율"] == 0), '부채비율'] = df_0_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 1)&(df["부채비율"] == 0), '부채비율'] = df_1_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 2)&(df["부채비율"] == 0), '부채비율'] = df_2_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 3)&(df["부채비율"] == 0), '부채비율'] = df_3_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 4)&(df["부채비율"] == 0), '부채비율'] = df_4_건설업
df.loc[(df.대업종 == '건설업')&(df.기업규모 == 5)&(df["부채비율"] == 0), '부채비율'] = df_5_건설업

df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 0)&(df["부채비율"] == 0), '부채비율'] = df_0_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 1)&(df["부채비율"] == 0), '부채비율'] = df_1_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 2)&(df["부채비율"] == 0), '부채비율'] = df_2_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 3)&(df["부채비율"] == 0), '부채비율'] = df_3_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 4)&(df["부채비율"] == 0), '부채비율'] = df_4_기타의사업
df.loc[(df.대업종 == '기타의사업')&(df.기업규모 == 5)&(df["부채비율"] == 0), '부채비율'] = df_5_기타의사업

df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 0)&(df["부채비율"] == 0), '부채비율'] = df_0_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 1)&(df["부채비율"] == 0), '부채비율'] = df_1_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 2)&(df["부채비율"] == 0), '부채비율'] = df_2_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 3)&(df["부채비율"] == 0), '부채비율'] = df_3_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 4)&(df["부채비율"] == 0), '부채비율'] = df_4_운수·창고·통신업
df.loc[(df.대업종 == '운수·창고·통신업')&(df.기업규모 == 5)&(df["부채비율"] == 0), '부채비율'] = df_5_운수·창고·통신업

In [95]:
df[df["부채비율"].isnull()]

,id,left,top,right,bottom,그리드 내 총 기업 수,연번,시도명,시군구명,관리기관,...,평균급여_2,평균급여_3,평균급여_4,평균급여_5,평균급여_6,위도,경도,재해발생년도,매출액,부채비율


In [97]:
df.to_excel("C:/Users/PC/Desktop/민재100.xlsx", index = False)